## Markov Chain Multi-Touch Attribution

## Goal

This notebook produces channel-level multi-touch attribution using a Markov Chain removal-effect method.

### What we build in this notebook

1. Load customer journeys and experiment outcomes from the ETL layer.
2. Convert per-customer journeys into ordered channel paths ending in absorbing states (`CONVERSION` or `NULL`).
3. Estimate the Markov transition matrix.
4. Compute baseline conversion probability of the chain.
5. Compute per-channel *removal effect* (incremental contribution).
6. Turn those into normalized attribution weights.
7. Join with spend to compute ROAS-style KPIs.
8. Persist final attribution outputs to the feature store (`data/outputs`).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import json

# Resolve project root similar to your Bayesian uplift notebook logic
CWD = Path().resolve()
if CWD.name in ["notebooks", "etl"]:
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

OUTPUT_DIR = PROJECT_ROOT / "etl" / "data" / "outputs"
print("CWD:", CWD)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Exists:", OUTPUT_DIR.exists())

CWD: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\notebooks
PROJECT_ROOT: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab
OUTPUT_DIR: C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\etl\data\outputs
Exists: True


### 1. Load inputs from ETL layer

We will pull:

- `sessions.parquet`: (customer_id, path) from the ETL notebook. `path` is a list of raw events like `view`, `addtocart`, `transaction`.
- `ab_experiment.parquet`: (customer_id, treatment, converted). We'll use `converted` as the purchase flag.
- `weekly_sales.parquet`: aggregated weekly sales (to get total conversions volume for ROAS).
- `channel_spend.parquet`: simulated spend per channel per week.

We'll stitch these into the inputs we need for attribution.

In [2]:
sessions_path = OUTPUT_DIR / "sessions.parquet"
ab_path       = OUTPUT_DIR / "ab_experiment.parquet"
sales_path    = OUTPUT_DIR / "weekly_sales.parquet"
spend_path    = OUTPUT_DIR / "channel_spend.parquet"

sessions_df = pd.read_parquet(sessions_path)
ab_df       = pd.read_parquet(ab_path)
weekly_sales_full = pd.read_parquet(sales_path)
channel_spend = pd.read_parquet(spend_path)

print("sessions_df.head():")
print(sessions_df.head())
print("\nab_df.head():")
print(ab_df.head())
print("\nweekly_sales_full.head():")
print(weekly_sales_full.head())
print("\nchannel_spend.head():")
print(channel_spend.head())

sessions_df.head():
  customer_id                      path
0           0        [view, view, view]
1           1                    [view]
2          10                    [view]
3         100  [view, view, view, view]
4        1000                    [view]

ab_df.head():
  customer_id  treatment  converted
0      257597          1      False
1      992329          1      False
2      111016          1      False
3      483717          1      False
4      951259          1      False

weekly_sales_full.head():
        week  sales
0 2015-04-27     83
1 2015-05-04   1103
2 2015-05-11   1218
3 2015-05-18   1061
4 2015-05-25   1146

channel_spend.head():
        week        search        social         email       display  \
0 2015-04-27  23789.287189  15266.425579  12104.222338  23274.454053   
1 2015-05-04  24156.294785  17341.734672  11490.335828  26142.268279   
2 2015-05-11  24527.969931  15965.130711  11305.326584  22723.271698   
3 2015-05-18  25932.831159  16290.584648  11225.056

### 2. Map raw events to marketing channels

The ETL `sessions_df` currently has a sequence of site events per `customer_id` (e.g. `view`, `addtocart`, `transaction`).

But attribution is channel-based, not event-based.

Because we don't have true channel tags in RetailRocket, we'll create a synthetic mapping:

- If the event contains certain keywords → map to a paid/owned channel.
- Else fallback to `direct`.

In [3]:
def map_event_to_channel(event_name: str) -> str:
    
    e = str(event_name).lower()

    # toy heuristics:
    if "email" in e:
        return "email"
    if "social" in e or "share" in e:
        return "social"
    if "search" in e:
        return "search"
    if "banner" in e or "display" in e or "ad" in e:
        return "display"

    # fallback
    return "direct"


def dedupe_consecutive(seq):
    """
    Collapse consecutive duplicates: [email, email, social] -> [email, social].
    This prevents self-loops from dominating transition counts.
    """
    out = []
    for s in seq:
        if not out or out[-1] != s:
            out.append(s)
    return out


# sessions_df has columns [customer_id, path]
# where path is a list of events like ["view","addtocart",...] from ETL

journeys = (
    sessions_df
    .merge(
        ab_df[["customer_id","converted"]],
        on="customer_id",
        how="left"
    )
    .fillna({"converted":0})
)

# Map each raw event in path -> channel, then dedupe consecutive
journeys["touches"] = journeys["path"].apply(
    lambda events: dedupe_consecutive([map_event_to_channel(e) for e in events])
)

print(journeys.head())
print("\nExample touches (channel sequence):")
for i,row in journeys.head(3).iterrows():
    print(row["customer_id"], row["touches"], "converted=", row["converted"])

  customer_id                      path  converted   touches
0           0        [view, view, view]      False  [direct]
1           1                    [view]      False  [direct]
2          10                    [view]      False  [direct]
3         100  [view, view, view, view]      False  [direct]
4        1000                    [view]      False  [direct]

Example touches (channel sequence):
0 ['direct'] converted= False
1 ['direct'] converted= False
10 ['direct'] converted= False


### 3. Add absorbing states (`START`, `CONVERSION`, `NULL`)

Markov attribution assumes that every path:

- starts at `START`
- ends at either:
  - `CONVERSION` (customer bought), or
  - `NULL` (customer churned / did not convert in window)

We'll now transform each customer's channel touch list → full Markov path:

`["START", ...channels..., "CONVERSION"]`  
or  
`["START", ...channels..., "NULL"]`

In [4]:
def finalize_path(touches, converted_flag):
    base = ["START"] + list(touches)
    if converted_flag:
        return base + ["CONVERSION"]
    else:
        return base + ["NULL"]

journeys["path_final"] = journeys.apply(
    lambda r: finalize_path(r["touches"], int(r["converted"])) , axis=1
)

print(journeys[["customer_id","path_final","converted"]].head())

# Sanity check that all paths start and end in correct states
assert all(p[0]=="START" for p in journeys["path_final"])
assert all(p[-1] in ("CONVERSION","NULL") for p in journeys["path_final"])

  customer_id             path_final  converted
0           0  [START, direct, NULL]      False
1           1  [START, direct, NULL]      False
2          10  [START, direct, NULL]      False
3         100  [START, direct, NULL]      False
4        1000  [START, direct, NULL]      False


### 4. Build transition matrix (Markov chain)

We'll:
1. Count all pairwise transitions `A -> B` across all customer paths.
2. Build a state list and a state index.
3. Create a transition probability matrix `P` such that `P[i,j] = P(next_state = j | current_state=i)`.

Notes:
- `CONVERSION` and `NULL` are absorbing (terminal). Their outgoing rows should be all zeros.
- We'll keep `P` as a NumPy array and also persist metadata for interpretability.

In [5]:
def build_transition_matrix(paths):
    """
    Given a list of paths (each a list of states),
    return:
      states           : sorted list of unique states
      state_index      : dict state->row/col index
      P                : transition prob matrix [n_states x n_states]
      trans_counts     : Counter of raw transition counts
    """
    trans_counts = Counter()
    states_set = set()

    for path in paths:
        for a,b in zip(path[:-1], path[1:]):
            trans_counts[(a,b)] += 1
            states_set.add(a)
            states_set.add(b)

    states = sorted(states_set)
    state_index = {s:i for i,s in enumerate(states)}

    P = np.zeros((len(states), len(states)), dtype=float)
    row_sums = defaultdict(int)
    for (a,b),count in trans_counts.items():
        i = state_index[a]
        j = state_index[b]
        P[i,j] += count
        row_sums[a] += count

    # row-normalize
    for s,i in state_index.items():
        if row_sums[s] > 0:
            P[i,:] /= row_sums[s]
        # absorbing states will have row_sums=0 and remain zeros.

    return states, state_index, P, trans_counts

states, state_index, P, trans_counts = build_transition_matrix(journeys["path_final"].tolist())

print("States:", states)
print("\nSample of transition probabilities from START:")
start_idx = state_index["START"]
for s2_idx, prob in enumerate(P[start_idx]):
    if prob>0:
        print("START ->", states[s2_idx], ":", round(prob,4))

States: ['CONVERSION', 'NULL', 'START', 'direct', 'display']

Sample of transition probabilities from START:
START -> direct : 0.9961
START -> display : 0.0039


### 5. Compute baseline conversion probability using absorbing Markov chains

We treat `CONVERSION` and `NULL` as absorbing states:

- Transient states: `START`, channels, etc.
- Absorbing states: `CONVERSION`, `NULL`.

We reorder states so all transient states come first, then absorbing. Then:

1. Partition `P` into:
   - `Q`: transient→transient
   - `R`: transient→absorbing
2. Compute fundamental matrix `N = (I - Q)^(-1)`.
3. Compute absorption probabilities `B = N @ R`.

`B[row_of_START, col_of_CONVERSION]` gives the probability of eventually converting (vs churning) according to this Markov chain.

We'll wrap this logic into a helper that returns:
- baseline probability of conversion starting at `START`
- diagnostic objects

In [6]:
def absorption_probability_convert(states, P, absorbing_names=("CONVERSION","NULL")):
    """
    Given states[] and transition matrix P, compute the probability that
    starting from START we eventually land in CONVERSION rather than NULL.

    Returns dict with:
      p_convert_baseline
      ordered_states
      transient_states
      absorbing_states
      B  (absorption probability matrix)
    """

    absorbing_set = set(absorbing_names)
    transient_states = [s for s in states if s not in absorbing_set]
    absorbing_states = [s for s in states if s in absorbing_set]

    ordered_states = transient_states + absorbing_states
    idx_ord = {s:i for i,s in enumerate(ordered_states)}

    # Reorder P into P_ord following ordered_states
    n = len(ordered_states)
    P_ord = np.zeros((n,n))
    orig_idx = {s:i for i,s in enumerate(states)}

    for a in ordered_states:
        for b in ordered_states:
            P_ord[idx_ord[a], idx_ord[b]] = P[orig_idx[a], orig_idx[b]]

    t = len(transient_states)
    a = len(absorbing_states)
    Q = P_ord[:t, :t]
    R = P_ord[:t, t:]

    I = np.eye(t)
    # Fundamental matrix
    N = np.linalg.inv(I - Q)
    # Absorption probabilities
    B = N @ R  # shape [t x a]

    # We assume START is transient and 'CONVERSION' is absorbing
    start_row = transient_states.index("START")
    conv_col = absorbing_states.index("CONVERSION")
    p_convert_baseline = B[start_row, conv_col]

    return {
        "p_convert_baseline": float(p_convert_baseline),
        "ordered_states": ordered_states,
        "transient_states": transient_states,
        "absorbing_states": absorbing_states,
        "B": B,
    }

baseline_results = absorption_probability_convert(states, P)
p_convert_baseline = baseline_results["p_convert_baseline"]
print("Baseline Pr(convert | START):", p_convert_baseline)

Baseline Pr(convert | START): 0.008325636908736982


### 6. Channel removal experiment (incremental contribution)

Here's the core of Markov attribution:

For each channel `c`:

1. Remove `c` from every path.
   - Example: `[START, email, social, search, CONVERSION]` with channel=`email` removed → `[START, social, search, CONVERSION]`.
2. If removing creates repeats (e.g. `[START, email, email, social]` without `email` → `[START, social]`), we naturally get shorter paths.
3. Recompute the Markov chain and the baseline conversion probability *without* that channel.
4. The removal effect for channel `c` is:

```text
lift[c] = p_convert_baseline - p_convert_without_c

In [7]:
def remove_channel_from_path(path, channel_to_remove):
    """
    Given a path like [START, email, social, CONVERSION],
    drop all occurrences of channel_to_remove.
    Keep START and terminal state (CONVERSION/NULL) intact.
    """
    if len(path) < 2:
        return path

    start = path[0]
    end   = path[-1]
    middle = [p for p in path[1:-1] if p != channel_to_remove]

    new_path = [start] + middle + [end]

    # Collapse consecutive duplicates after removal
    deduped = []
    for s in new_path:
        if not deduped or deduped[-1] != s:
            deduped.append(s)

    # Edge case: if deduped == [START] only, re-attach terminal state
    if len(deduped) == 1:
        deduped = [start, end]

    return deduped


def channel_removal_effect(journeys_paths, channel):
    """
    Compute conversion probability after removing `channel`,
    then return that probability + debug info.
    """
    new_paths = [remove_channel_from_path(p, channel) for p in journeys_paths]
    st, st_idx, P_new, tc_new = build_transition_matrix(new_paths)
    res = absorption_probability_convert(st, P_new)
    return res["p_convert_baseline"], {
        "states": st,
        "P": P_new,
        "trans_counts": tc_new,
        "details": res,
    }

# List of candidate marketing channels we want to evaluate.
# We'll infer from all non-absorbing states except START, assuming those are channels.
absorbing_like = {"START","CONVERSION","NULL"}
candidate_channels = [s for s in states if s not in absorbing_like]
print("Candidate channels:", candidate_channels)

channel_lift = {}
channel_details = {}

all_paths = journeys["path_final"].tolist()

for ch in candidate_channels:
    p_without_ch, debug_info = channel_removal_effect(all_paths, ch)
    lift = p_convert_baseline - p_without_ch
    channel_lift[ch] = float(lift)
    channel_details[ch] = debug_info

print("\nBaseline conversion prob:", p_convert_baseline)
print("Channel removal lifts:")
for ch, lift in channel_lift.items():
    print(f"  {ch}: {lift:.6f}")

Candidate channels: ['direct', 'display']

Baseline conversion prob: 0.008325636908736982
Channel removal lifts:
  direct: 0.000000
  display: 0.000000


### 7. Normalize lifts → attribution weights

Raw `lift[ch]` is "how much baseline conversion probability drops if we delete channel `ch`".

Next we convert those into channel attribution shares that sum to 1:

```python
share[ch] = lift[ch] / sum(max(lift[c],0) for c in channels)

In [8]:
total_positive_lift = sum(v for v in channel_lift.values() if v>0)
if total_positive_lift <= 0:
    attribution_share = {ch:0.0 for ch in channel_lift}
else:
    attribution_share = {
        ch: (v/total_positive_lift if v>0 else 0.0)
        for ch,v in channel_lift.items()
    }

print("Attribution share (sums ~1):")
print(attribution_share)

# 2) total conversions volume.
# We'll define total conversions as total observed purchases in weekly_sales_full.
# (In a real pipeline you'd use actual transaction counts or revenue.)
total_conversions = float(weekly_sales_full["sales"].sum())
print("\nTotal conversions in period:", total_conversions)

channel_conversions = {
    ch: attribution_share[ch] * total_conversions
    for ch in attribution_share
}

print("\nChannel conversions (allocated):")
for ch,cv in channel_conversions.items():
    print(f"  {ch}: {cv:.2f}")

# 3) Spend totals per channel across weeks
# channel_spend columns: [week, search, social, email, display, direct]
spend_totals = (
    channel_spend.drop(columns=["week"], errors="ignore")
                 .sum(numeric_only=True)
                 .to_dict()
)

print("\nSpend totals:")
for ch,sp in spend_totals.items():
    print(f"  {ch}: {sp:.2f}")

# 4) ROAS-like metric: conversions per dollar spend
roas = {}
for ch in attribution_share:
    spend = spend_totals.get(ch, 0.0)
    if spend <= 0:
        roas[ch] = np.nan
    else:
        roas[ch] = channel_conversions[ch] / spend

print("\nROAS-style (conversions per $1 spend):")
for ch,val in roas.items():
    print(f"  {ch}: {val}")

Attribution share (sums ~1):
{'direct': 1.0, 'display': 0.0}

Total conversions in period: 22457.0

Channel conversions (allocated):
  direct: 22457.00
  display: 0.00

Spend totals:
  search: 567117.45
  social: 363244.25
  email: 197893.22
  display: 438806.01
  direct: 627432.27

ROAS-style (conversions per $1 spend):
  direct: 0.035791910961383865
  display: 0.0


### 8. Persist outputs

We'll persist two things for downstream consumers:

1. `markov_attribution.parquet` — a tidy table for BI / dashboards.

   Columns:
   - `channel`
   - `lift` (removal effect)
   - `attribution_share`
   - `attributed_conversions`
   - `spend`
   - `roas_like`

2. `markov_summary.json` — lightweight summary for APIs / alerting.

   Fields:
   - baseline conversion prob from chain
   - total_conversions for the period
   - dict of channel shares
   - dict of ROAS / efficiency

In [9]:
# 1. Build attribution DataFrame
attrib_records = []
for ch in attribution_share:
    attrib_records.append({
        "channel": ch,
        "lift": channel_lift.get(ch, np.nan),
        "attribution_share": attribution_share.get(ch, np.nan),
        "attributed_conversions": channel_conversions.get(ch, np.nan),
        "spend": spend_totals.get(ch, np.nan),
        "roas_like": roas.get(ch, np.nan)
    })

attrib_df = pd.DataFrame(attrib_records)
attrib_df = attrib_df.sort_values("attribution_share", ascending=False)

print("Markov attribution table:")
print(attrib_df)

# 2. Write parquet
attrib_out_path = OUTPUT_DIR / "markov_attribution.parquet"
attrib_df.to_parquet(attrib_out_path, index=False)
print(f"\nWrote {attrib_out_path}")

# 3. Build summary JSON
summary_obj = {
    "baseline_conversion_prob": p_convert_baseline,
    "total_conversions": total_conversions,
    "channel_attribution_share": attribution_share,
    "channel_lift": channel_lift,
    "channel_conversions": channel_conversions,
    "channel_spend": spend_totals,
    "channel_roas_like": {
        k:(None if (isinstance(v,float) and np.isnan(v)) else v)
        for k,v in roas.items()
    },
}

summary_path = OUTPUT_DIR / "markov_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_obj, f, indent=2)

print(f"Wrote {summary_path}")

Markov attribution table:
   channel          lift  attribution_share  attributed_conversions  \
0   direct  1.734723e-18                1.0                 22457.0   
1  display  0.000000e+00                0.0                     0.0   

           spend  roas_like  
0  627432.271617   0.035792  
1  438806.006738   0.000000  

Wrote C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\etl\data\outputs\markov_attribution.parquet
Wrote C:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\etl\data\outputs\markov_summary.json


### How efficiently are they spending?

In [10]:
top_msg_lines = []
top_msg_lines.append(
    f"Baseline conversion probability (Markov): {p_convert_baseline:.3f}"
)
top_msg_lines.append(
    f"Total conversions in period: {total_conversions:.0f}"
)

top_msg_lines.append("\nChannel contribution (share of incremental conversions):")
for _,row in attrib_df.iterrows():
    ch = row["channel"]
    share = row["attribution_share"]
    convs = row["attributed_conversions"]
    spend = row["spend"]
    roi   = row["roas_like"]
    top_msg_lines.append(
        f" - {ch}: {share:.1%} of incremental lift, ~{convs:.1f} convs, "
        f"spend ${spend:,.0f}, ROAS-like {roi:.4f} conv/$"
    )

report_text = "\n".join(top_msg_lines)
print(report_text)

Baseline conversion probability (Markov): 0.008
Total conversions in period: 22457

Channel contribution (share of incremental conversions):
 - direct: 100.0% of incremental lift, ~22457.0 convs, spend $627,432, ROAS-like 0.0358 conv/$
 - display: 0.0% of incremental lift, ~0.0 convs, spend $438,806, ROAS-like 0.0000 conv/$
